# F1: 시간 가드 융합 (어려운 한글 행 → v20, 나머지 → 112)
**GPU 런타임(L4 권장)**으로 위에서부터 실행하세요. 마지막에 `F1: PASS`가 나오면 `combo_kit/out/F1.zip`이 생깁니다. `FAIL`이면 올리지 마세요.
먼저 DACON 제출 목록에서 **115(C_vote)의 서버 실행 시간(분)**을 확인해 아래 `T_c`에 넣으세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la build_fuse_safe.py v20.zip 112_submit.zip check_combo.py
!nvidia-smi --query-gpu=name --format=csv
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2


In [ ]:
T_c = 15.0   # <- 115(C_vote)의 서버 실행 시간(분)
BUD = int(min(24, 28.5 - max(0.0, T_c - 11.8)))
BUD = max(BUD, 18)
print("v20 시간 예산(분):", BUD)
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)


In [ ]:
# 조립 (약 15분)
zp = f"{W}/F1.zip"
if os.path.exists(zp): os.remove(zp)
!unzip -p v20.zip script.py | grep -n "def run"
!python build_fuse_safe.py {zp} v20.zip 112_submit.zip 4 {BUD}


In [ ]:
# 점검 (약 20분): 3,000행 실행 + 영문 불변 + 600행 행 독립성. 통과할 때만 out/F1.zip으로 복사
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_F1"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/F1_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "router start", "OK", "PROBLEM", "WARNING ko", "WARNING en")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING en", "still running", "did not start", "en failed") if k in log]
last = (c.stdout.strip().splitlines() or [""])[-1]
ok = c.returncode == 0 and last == "OK" and not bad and log.count("fell back 0,") == 2 and os.path.getsize(zp) < 10e9
if ok:
    shutil.copy(zp, f"{D}/out/F1.zip"); print("F1: PASS ->", f"{D}/out/F1.zip", round(os.path.getsize(f"{D}/out/F1.zip")/1e9, 2), "GB")
else:
    print("F1: FAIL - 올리지 마세요. 문제:", bad or last, " 전체 로그:", f"{W}/F1_check.log")
